# Machine Learning Pipelines and Orchestration with Prefect

> ⚠️  **Version**: This module uses the course's pinned Prefect 2.13.7. Run `uv sync` and use the project kernel. Download the taxi data using the introduction notebook first. The completed implementation lives in `lib/`; this notebook imports and exercises that same code instead of maintaining a second copy. Without `PREFECT_API_URL`, Prefect uses its local ephemeral API. To see runs in the UI, start the server and set the API URL before running the kernel.

## 0 - Useful functions

### 0.1 - From previous lessons


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
LESSON_FOLDER = REPO_ROOT / "lessons" / "03-pipeline-and-orchestration"
if str(LESSON_FOLDER) not in sys.path:
    sys.path.insert(0, str(LESSON_FOLDER))

from lib.config import ARTIFACTS_FILEPATH, INPUT_FILEPATH, PREDICTIONS_FILEPATH, TEST_FILEPATH, TRAIN_FILEPATH  # noqa: E402

for filepath in (TRAIN_FILEPATH, TEST_FILEPATH, INPUT_FILEPATH):
    assert Path(filepath).is_file(), f"Download the introduction data first: {filepath}"

In [ ]:
import inspect

from lib.preprocessing import process_data

# Inspect the task/subflow implementation shared with the executable scripts.
print(inspect.getsource(process_data.fn))

### 0-2 Helpers for this session

You also have other helpers to show you prefect's features in the `helpers.py` file.

In [ ]:
from lib.helpers import save_pickle

# Atomic bundle writes keep a weekly training update from exposing a partial model.
print(inspect.getsource(save_pickle.fn))

## 1 - Create workflow functions

Create five functions to complete the ML process :
- `train_model`
- `predict`
- `evaluate_model`
- A workflow function to perform the whole training process `train_model_workflow`
    - Process data
    - Train model
    - Evaluate model
- A workflow function to perform the whole prediction process `batch_predict_workflow`
    - Process data without target column
    - Predict


For the last two functions, you can start without saving / loading artifacts add these steps after.
Please think about what artifacts you'll need to save and load to pass from training to predict workflows.

Start by coding these functions here in the notebook

Then, test your code with the downloaded data (e.g. January to train and February to predict).

Finally, copy your code in the `lib` folder in the `modeling.py` and `workflows.py` files and test your workflows again using such a command:

```bash
uv run python -m lib.workflows train
uv run python -m lib.workflows predict
```

In [ ]:
import numpy as np
from lib.workflows import batch_predict_workflow, train_model_workflow

# The model and its fitted DictVectorizer must be persisted together.
training = train_model_workflow(TRAIN_FILEPATH, TEST_FILEPATH, artifacts_filepath=ARTIFACTS_FILEPATH)
assert training["test_rmse"] < training["baseline_test_rmse"]
{key: training[key] for key in ("train_rmse", "test_rmse", "baseline_test_rmse", "feature_count")}

## 2 - Setup and explore Prefect

We are going to use [Prefect](https://docs.prefect.io/2.6/tutorials/first-steps/), an Open Source orchestration tool with a Python SDK.


**WINDOWS USERS**:

You might run into issues with Prefect on Windows. If you do, please follow [Prefects instructions](https://docs.prefect.io/v3/get-started/install) to install Prefect on your machine

### 2-1 Setup Prefect UI

Before starting to implement tasks and flows with prefect, let's set up the UI in order to have a good visualization of our work.

Steps :

- Set an API URL for your local server to make sure that your workflow will be tracked by this specific instance :
```
export PREFECT_API_URL=http://127.0.0.1:4200/api
```

- Check you have SQLite installed ([Prefect backend database system](https://docs.prefect.io/2.13.7/getting-started/installation/#external-requirements)):
```
sqlite3 --version 
```

- Start a local prefect server :
```
uv run prefect server start --host 127.0.0.1
```

Use the same API URL in the terminal that launches the kernel and the deployment runner. No database reset is needed. With no API URL, Run All still works against Prefect's ephemeral local API.


You can visit the UI at http://127.0.0.1:4200/dashboard

![](images/starting_page.png)

### 2-2 Prefect tasks and flows

[Prefect uses tasks and flows to build workflows](https://docs.prefect.io/2.13.7/tutorial/flows/).
- Flows are like functions. They can take inputs, perform work, and return an output. In fact, you can turn any function into a Prefect flow by adding the @flow decorator
- A task is any Python function decorated with a @task decorator called within a flow. You can think of a flow as a recipe for connecting a known sequence of tasks together. Tasks, and the dependencies between them, are displayed in the flow run graph, enabling you to break down a complex flow into something you can observe, understand and control at a more granular level.
    - All tasks must be called from within a flow. Tasks may not call other tasks directly.
    - Not all functions in a flow need be tasks. Use them only when their features are useful.

In [ ]:
# Example
import httpx
from prefect import flow, task


@task
def get_url(url: str, params: dict | None = None) -> dict:
    """Get the a response from an URL call."""
    response = httpx.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


@flow(retries=3, retry_delay_seconds=5, log_prints=True)
def get_repo_info(repo_name: str = "PrefectHQ/prefect") -> None:
    """Get number of stars for a GitHub Repo."""
    url = f"https://api.github.com/repos/{repo_name}"
    repo_stats = get_url(url)
    print(f"{repo_name} repository statistics 🤓:")
    print(f"Stars 🌠 : {repo_stats['stargazers_count']}")
    print(f"Forks 🍴 : {repo_stats['forks_count']}")

## 3 - Create Prefect tasks and flows

### 3-1 Create tasks and flows

Use the decorators `@task` and `@flow` to create your first prefect flow : The Processing flow.

Prefect will try to use by default different thread to run each task. If you want sequential steps, introduce this dependencies through the name of each task output.


Steps:
- Create a task for each function you created in the previous section. You can start by doing these in the notebook.
- Test your code by calling the flows run with downloaded data (this can be done in the notebook too).
- Update your files in the `lib` folder. You should now have completed all files except `deployment.py`.


You can see registered flows in the UI :
![Flows in Prefect UI](images/flows_ui.png)


And visualize the run of a flow :
![Flows in Prefect UI](images/flow_run.png)


> [!Warning]
> **Typing tasks and flows in prefect** :
> Typing tasks in prefect is done as with any python code.
> For flows, either use `validate_parameters=False` or define pydantic models for prefect to understand your NON DEFAULT typing (see extra section).
> But if all tasks are typed, since flows are just set of tasks, it should be all good if we don't want to add a layer of complexity
> `Default types` : str, int ...




### Completed workflow check

Run batch prediction independently using the saved bundle. Inference skips target computation and outlier filtering, so every source row keeps its position, even when no timestamps are available. The fitted vectorizer is loaded, never refitted.

In [ ]:
predictions = batch_predict_workflow(INPUT_FILEPATH, artifacts_filepath=ARTIFACTS_FILEPATH, output_filepath=PREDICTIONS_FILEPATH)
assert np.isfinite(predictions).all()
np.testing.assert_array_equal(predictions, np.load(PREDICTIONS_FILEPATH, allow_pickle=False))
print(f"Saved {len(predictions):,} ordered predictions.")
print(f"Outside [1, 60] minutes: {((predictions < 1) | (predictions > 60)).sum():,}")

### 3-2 Customize your flows

The implementation uses named, tagged tasks and retries transient file I/O twice with a one-second delay. Deterministic preprocessing/model errors are not retried. Large in-memory task results are not persisted by Prefect; the explicit atomic model bundle is the durable handoff between scheduled training and prediction. Flow logs report RMSE, row counts and implausible predictions.

You can configure the properties and special behavior for your prefect tasks/flow in the decorator.
For example, you can tell if you want to retry on a failure, set name or tags, etc... \
An example is given in the `helpers.py` file.
```
@task(name='failure_task', tags=['fails'], retries=3, retry_delay_seconds=60)
def func():
  ...

```

- Add names, tasks, and desired behavior to your tasks/flows
- Test your code
- Visualize in the local prefect UI

If a task fails in the flow, it is possible to visualize which task fail and access the full log and traceback error
by clicking on the tasks. \
We can also access run information inside de `state` object that can be returned by the flows using python code

## 4 - Deploy your flows

Now that all the workflows are defined, we can now schedule automatics runs for these pipelines.
Let's assume that we have a process that tells us that our model need to be retrained weekly based on some performance analysis. We also receive data to predict each hour.

Use prefect deployment object in order to :
- Schedule complete ml process to run weekly
- Schedule prediction pipeline to run each hour


**Please note that you can test your code with the `to_deployment` here, however you'll have to move to scripts to test the deployment with `serve`.**

You can deploy your flows by following [Prefect documentation here](https://docs.prefect.io/2.13.7/tutorial/deployments/#running-multiple-deployments-at-once).

⚠️  Serving a model with prefect is a long-running command, meaning that you will need to run it in a separate terminal or in the background.
Interupting the command will stop the deployment, but you'll be still be able to see it the UI.

In the UI, you should be able see deployments:
![Deployments in Prefect UI](images/deployments.png)


And the scheduled runs for one deployment:
![Scheduled runs in Prefect UI](images/scheduled_runs.png)

In [ ]:
from lib.deployment import build_deployments

weekly, hourly = await build_deployments()
assert weekly.schedule.cron == "0 0 * * 1"
assert hourly.schedule.cron == "0 * * * *"
assert weekly.schedule.timezone == hourly.schedule.timezone == "UTC"
for deployment in (weekly, hourly):
    print(deployment.name, deployment.schedule.json())
# Start `uv run python -m lib.deployment` in a separate terminal to register
# and serve both deployments. Never call the blocking serve() from Run All.

## 5 - Extra concepts

### 5-1 Prefect Work-Pools and Workers

As you might deal with large datasets, training and infering using your local machine will be impractical or simply impossible.

You will need to use the computational power offered by cloud providers like Google Cloud, Microsoft Azure or Amazon Web Services (running a flow using a Docker container is also an option).

Work pools are a bridge between the Prefect orchestration layer and the infrastructure where flows are run. It allows to dynamically define the infrastructure in which you will run your workflow.

Workers are then the object that retrieve scheduled runs from a work pool and execute them.

You can learn more [here](https://docs.prefect.io/v3/concepts/work-pools)

You can try to run your workflow using Docker ([doc](https://docs.prefect.io/v3/how-to-guides/deployment_infra/serve-flows-docker))

### 5-2 Prefect typing using Pydantic

By default, Prefect validate the typing of the functions' arguments for base types (str, int...).

However, Prefect flows often pass complex parameters and artifacts between tasks or across deployment boundaries — and without proper validation, bugs can silently propagate.

Pydantic helps ensure that:

- Inputs to flows and tasks are validated at runtime (e.g., wrong types, missing fields)

- Data structures are clearly defined and typed, improving code readability and maintainability

- You catch errors early, before they cause downstream failures in your pipeline


You can learn more [here](https://docs.prefect.io/v3/how-to-guides/workflows/pass-inputs#validate-inputs) and [here](https://www.prefect.io/blog/data-validation-with-pydantic)

## Completed results

- The full January/February training flow completed with **528 one-hot features**, training RMSE **6.7824 minutes**, and test RMSE **7.3711 minutes**, versus **9.2953 minutes** for the training-mean baseline. The same training-fitted vectorizer and duration filter were used for evaluation.
- The independent March prediction flow loaded the saved model/vectorizer bundle and saved **1,925,152** finite, ordered predictions. The saved NumPy output exactly matched the returned array. **3,809** raw estimates fall outside the 1–60 minute training range; these are logged, not silently clipped.
- Weekly training is scheduled for Monday 00:00 UTC; batch prediction runs at minute 0 each hour. The notebook builds these deployment definitions without blocking. In a separate isolated local server test, both deployments registered and a prediction run submitted through the runner reached **Completed**.
- Runner shutdown was verified to pause both schedules before the temporary server was stopped. No unattended scheduled jobs were left running. Tests additionally exercise missing/unknown categories, target-free input, matching in-memory/saved predictions, failed flow states, atomic artifact updates and schedule definitions.

This is a local orchestration demonstration, not a continuously refreshed production pipeline: the schedules reuse historical files and replace the same prediction output. Production use would need fresh-input discovery, versioned outputs, validation/promotion rules and a policy for implausible estimates. See the README for execution, UI, runner and test commands. Generated artifacts and notebook outputs are excluded from Git.